# Auditoría de resultados

Este notebook **no cambia** los notebooks 01 a 04: vuelve a calcular las cifras de la presentación y agrega las pruebas
que faltaban. Usa el mismo dataset procesado, la misma separación en entrenamiento y test y los mismos hiperparámetros
que el notebook 02.

1. Crecimiento de Vaca Muerta: cuánto viene de sumar pozos y cuánto del caudal.
2. Test final **sin días efectivos** (`tef`): con los mismos hiperparámetros y re-optimizados sólo con entrenamiento.
3. Importancia por permutación en una validación separada por pozo, para comparar con `feature_importances_`.
4. Concordancia K-Means / DBSCAN con el índice de Rand ajustado (ARI).

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.cluster import DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import adjusted_rand_score, f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.float_format", "{:,.3f}".format)
pet = pd.read_csv("../data/processed/pozos_petroleros_2025.csv.gz")

## 1. Crecimiento de Vaca Muerta

Producción del mes = pozos activos × días por pozo × caudal por día. Comparamos diciembre con enero y separamos
cohortes: pozos que produjeron en los dos meses y pozos que entraron después de enero. Con un solo año no se puede
saber si un pozo que entra es **nuevo** o **reactivado**; como aproximación miramos si tenía alguna declaración en enero
en el dataset completo.

In [2]:
vm = pet[pet["formprod"] == "VMUT"]
activos = vm[vm["tef"] > 0]
mes = activos.groupby("mes").agg(petroleo=("prod_pet", "sum"), pozos=("idpozo", "nunique"), dias=("tef", "sum"))
mes["dias_por_pozo"] = mes["dias"] / mes["pozos"]
mes["caudal_agregado"] = mes["petroleo"] / mes["dias"]
mes["caudal_mediano"] = activos.assign(q=activos["prod_pet"] / activos["tef"]).groupby("mes")["q"].median()
mes["petroleo_por_pozo"] = mes["petroleo"] / mes["pozos"]
variacion = mes.loc[12] / mes.loc[1] - 1
factores = ["pozos", "dias_por_pozo", "caudal_agregado"]
aporte_log = np.log(mes.loc[12, factores] / mes.loc[1, factores]) / np.log(mes.loc[12, "petroleo"] / mes.loc[1, "petroleo"])
pd.DataFrame({"variación dic/ene": variacion[["petroleo"] + factores + ["caudal_mediano", "petroleo_por_pozo"]],
              "% del crecimiento (log)": (aporte_log * 100).reindex(["petroleo"] + factores + ["caudal_mediano", "petroleo_por_pozo"])})

,variación dic/ene,% del crecimiento (log)
petroleo,0.351,NaN
pozos,0.209,63.033
dias_por_pozo,0.023,7.449
caudal_agregado,0.093,29.518
caudal_mediano,-0.013,NaN
petroleo_por_pozo,0.118,NaN


In [3]:
ene = activos[activos["mes"] == 1].set_index("idpozo")["prod_pet"]
dic = activos[activos["mes"] == 12].set_index("idpozo")["prod_pet"]
ambos, entran, salen = ene.index.intersection(dic.index), dic.index.difference(ene.index), ene.index.difference(dic.index)
raw = pd.read_csv("../data/produccion_pozos_2025.csv.gz", usecols=["idpozo", "mes", "tipoestado"], low_memory=False)
en_enero = raw.loc[(raw["mes"] == 1) & raw["idpozo"].isin(entran)]
pd.Series({
    "pozos en enero y diciembre": len(ambos),
    "  petróleo en enero de esos pozos (m³)": ene[ambos].sum(),
    "  petróleo en diciembre de esos pozos (m³)": dic[ambos].sum(),
    "pozos que entran después de enero": len(entran),
    "  sin ninguna declaración en enero (probablemente nuevos)": len(entran) - en_enero["idpozo"].nunique(),
    "  declarados en enero (reactivados)": en_enero["idpozo"].nunique(),
    "  petróleo en diciembre de los que entran (m³)": dic[entran].sum(),
    "pozos que salen": len(salen),
    "aumento total dic − ene (m³)": dic.sum() - ene.sum(),
}).round(0)

pozos en enero y diciembre                                      1,578.000
  petróleo en enero de esos pozos (m³)                      1,951,564.000
  petróleo en diciembre de esos pozos (m³)                  1,307,550.000
pozos que entran después de enero                                 430.000
  sin ninguna declaración en enero (probablemente nuevos)         355.000
  declarados en enero (reactivados)                                75.000
  petróleo en diciembre de los que entran (m³)              1,382,514.000
pozos que salen                                                    83.000
aumento total dic − ene (m³)                                  699,172.000
dtype: float64

**Lectura.** La mediana del caudal no se mueve, pero el caudal agregado sube 9 % y el petróleo por pozo 12 %. Unos dos
tercios del crecimiento (en escala log) corresponden a más pozos activos. Los pozos que producían en enero **declinan**
y los que entran durante el año compensan esa caída y explican todo el aumento. No podemos separar con certeza los pozos
nuevos de los reactivados.

## 2. Modelo supervisado: test con y sin días efectivos

Misma separación (GroupKFold con `random_state=42`) y mismos hiperparámetros que el notebook 02. El test se usa sólo
para medir; cuando se re-optimiza sin `tef`, Optuna usa los mismos 3.000 pozos de entrenamiento que en el notebook 02.

In [4]:
numericas, sin_tef = ["profundidad", "tef", "mes"], ["profundidad", "mes"]
categoricas = ["cuenca", "provincia", "tipo_de_recurso", "sub_tipo_recurso", "tipoextraccion", "formprod", "empresa"]
X, grupos = pet[numericas + categoricas], pet["idpozo"]
idx_train, idx_test = next(GroupKFold(n_splits=5, shuffle=True, random_state=42).split(X, groups=grupos))
X_train, X_test, g_train = X.iloc[idx_train], X.iloc[idx_test], grupos.iloc[idx_train]
prod_train, prod_test = pet["prod_pet"].iloc[idx_train], pet["prod_pet"].iloc[idx_test]

rng = np.random.default_rng(42)
pozos_muestra = rng.choice(g_train.unique(), size=6000, replace=False)
en_muestra = g_train.isin(pozos_muestra).to_numpy()
pozos_optuna = rng.choice(pozos_muestra, size=3000, replace=False)
en_optuna = g_train[en_muestra].isin(pozos_optuna).to_numpy()
X_optuna, g_optuna = X_train[en_muestra][en_optuna], g_train[en_muestra][en_optuna]

y_train, y_test = np.log1p(prod_train), np.log1p(prod_test)
_, cortes = pd.qcut(prod_train, q=3, retbins=True)
cortes[0], cortes[-1] = -np.inf, np.inf
c_train = pd.cut(prod_train, bins=cortes, labels=["Baja", "Media", "Alta"])
c_test = pd.cut(prod_test, bins=cortes, labels=["Baja", "Media", "Alta"])

# Hiperparámetros elegidos por Optuna en el notebook 02 (con tef)
HP_REG = {"n_estimators": 150, "max_depth": 16, "min_samples_leaf": 13, "max_features": 0.3}
HP_CLF = {"n_estimators": 50, "max_depth": 17, "min_samples_leaf": 18, "max_features": "sqrt"}


def modelo(algoritmo, num, escalar=False):
    pasos = [("imputar", SimpleImputer(strategy="median"))] + ([("escalar", StandardScaler())] if escalar else [])
    prep = ColumnTransformer([("num", Pipeline(pasos), num),
                              ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=500,
                                                    sparse_output=False), categoricas)])
    return Pipeline([("prep", prep), ("modelo", algoritmo)])


def test_regresion(pipe, num):
    pipe.fit(X_train[num + categoricas], y_train)
    pred = pipe.predict(X_test[num + categoricas])
    mae = mean_absolute_error(y_test, pred)
    return {"R² (log)": r2_score(y_test, pred), "MAE (log)": mae, "factor e^MAE": np.exp(mae),
            "% de filas con error ≤ factor": np.mean(np.abs(y_test - pred) <= mae) * 100}


def test_clasificacion(pipe, num):
    pipe.fit(X_train[num + categoricas], c_train)
    pred, proba = pipe.predict(X_test[num + categoricas]), pipe.predict_proba(X_test[num + categoricas])
    return {"F1 macro": f1_score(c_test, pred, average="macro"), "accuracy": np.mean(pred == c_test),
            "AUC (one-vs-rest, macro)": roc_auc_score(c_test, proba, multi_class="ovr", average="macro",
                                                      labels=pipe.classes_)}

In [5]:
def optimizar(Algoritmo, y, scoring, num):
    def objetivo(trial):
        hp = {"n_estimators": trial.suggest_int("n_estimators", 50, 150, step=50),
              "max_depth": trial.suggest_int("max_depth", 5, 30),
              "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50, log=True),
              "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.3, 0.6, 1.0])}
        return cross_val_score(modelo(Algoritmo(**hp, random_state=42, n_jobs=-1), num), X_optuna[num + categoricas],
                               y[en_muestra][en_optuna], groups=g_optuna, cv=GroupKFold(3), scoring=scoring).mean()
    estudio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    estudio.optimize(objetivo, n_trials=20)
    return estudio.best_params


HP_REG_SIN = optimizar(RandomForestRegressor, y_train, "r2", sin_tef)
HP_CLF_SIN = optimizar(RandomForestClassifier, c_train, "f1_macro", sin_tef)
HP_REG_SIN, HP_CLF_SIN

({'n_estimators': 150,
  'max_depth': 20,
  'min_samples_leaf': 13,
  'max_features': 0.3},
 {'n_estimators': 100,
  'max_depth': 29,
  'min_samples_leaf': 15,
  'max_features': 'sqrt'})

In [6]:
regresion = pd.DataFrame({
    "RF, con tef": test_regresion(modelo(RandomForestRegressor(**HP_REG, random_state=42, n_jobs=-1), numericas), numericas),
    "RF, sin tef (mismos hiperparámetros)": test_regresion(modelo(RandomForestRegressor(**HP_REG, random_state=42, n_jobs=-1), sin_tef), sin_tef),
    "RF, sin tef (re-optimizado)": test_regresion(modelo(RandomForestRegressor(**HP_REG_SIN, random_state=42, n_jobs=-1), sin_tef), sin_tef),
    "Lineal, con tef": test_regresion(modelo(LinearRegression(), numericas, escalar=True), numericas),
    "Lineal, sin tef": test_regresion(modelo(LinearRegression(), sin_tef, escalar=True), sin_tef),
}).T
regresion

,R² (log),MAE (log),factor e^MAE,% de filas con error ≤ factor
"RF, con tef",0.706,0.652,1.919,61.529
"RF, sin tef (mismos hiperparámetros)",0.535,0.841,2.319,62.082
"RF, sin tef (re-optimizado)",0.540,0.833,2.300,62.492
"Lineal, con tef",0.624,0.752,2.121,61.671
"Lineal, sin tef",0.435,0.932,2.539,62.191


In [7]:
clasificacion = pd.DataFrame({
    "RF, con tef": test_clasificacion(modelo(RandomForestClassifier(**HP_CLF, random_state=42, n_jobs=-1), numericas), numericas),
    "RF, sin tef (mismos hiperparámetros)": test_clasificacion(modelo(RandomForestClassifier(**HP_CLF, random_state=42, n_jobs=-1), sin_tef), sin_tef),
    "RF, sin tef (re-optimizado)": test_clasificacion(modelo(RandomForestClassifier(**HP_CLF_SIN, random_state=42, n_jobs=-1), sin_tef), sin_tef),
    "Logística, con tef": test_clasificacion(modelo(LogisticRegression(max_iter=1000), numericas, escalar=True), numericas),
    "Logística, sin tef": test_clasificacion(modelo(LogisticRegression(max_iter=1000), sin_tef, escalar=True), sin_tef),
}).T
clasificacion

,F1 macro,accuracy,"AUC (one-vs-rest, macro)"
"RF, con tef",0.614,0.613,0.801
"RF, sin tef (mismos hiperparámetros)",0.579,0.580,0.771
"RF, sin tef (re-optimizado)",0.581,0.582,0.773
"Logística, con tef",0.601,0.599,0.789
"Logística, sin tef",0.559,0.562,0.754


**Sobre el factor e^MAE.** Es una lectura multiplicativa del error absoluto medio en escala log, **no** un intervalo de
confianza ni de predicción. La columna "% de filas con error ≤ factor" muestra qué parte del test cae dentro de ese
factor: no es una garantía para un pozo puntual.

## 3. Importancia por permutación

`feature_importances_` mide la reducción de impureza y puede favorecer a las variables continuas o con muchas
divisiones posibles. Como control, entrenamos con el 80 % de los pozos de entrenamiento y permutamos cada variable
original en el 20 % restante (20.000 filas, 5 repeticiones). El test no se usa.

In [8]:
rf = modelo(RandomForestRegressor(**HP_REG, random_state=42, n_jobs=-1), numericas).fit(X_train, y_train)
nombres = rf.named_steps["prep"].get_feature_names_out()
variable = [next(v for v in numericas + categoricas if n.split("__", 1)[1].startswith(v)) for n in nombres]
impureza = pd.Series(rf.named_steps["modelo"].feature_importances_, index=nombres).groupby(variable).sum()

i_fit, i_val = next(GroupKFold(5, shuffle=True, random_state=7).split(X_train, groups=g_train))
rf_val = modelo(RandomForestRegressor(**HP_REG, random_state=42, n_jobs=-1), numericas)
rf_val.fit(X_train.iloc[i_fit], y_train.iloc[i_fit])
filas = np.random.default_rng(0).choice(len(i_val), 20000, replace=False)
perm = permutation_importance(rf_val, X_train.iloc[i_val].iloc[filas], y_train.iloc[i_val].iloc[filas],
                              scoring="r2", n_repeats=5, random_state=42)
pd.DataFrame({"impureza (agrupada)": impureza,
              "permutación: caída de R²": pd.Series(perm.importances_mean, index=X_train.columns),
              "permutación: desvío": pd.Series(perm.importances_std, index=X_train.columns)}
             ).sort_values("permutación: caída de R²", ascending=False)

,impureza (agrupada),permutación: caída de R²,permutación: desvío
tef,0.380,0.556,0.005
tipoextraccion,0.160,0.232,0.003
formprod,0.080,0.123,0.001
profundidad,0.163,0.100,0.001
sub_tipo_recurso,0.072,0.048,0.001
provincia,0.050,0.044,0.000
empresa,0.046,0.035,0.001
tipo_de_recurso,0.037,0.025,0.001
cuenca,0.009,0.006,0.000
mes,0.003,0.001,0.000


## 4. ¿Coinciden K-Means y DBSCAN?

Mismas variables y escalado que el notebook 03. El ARI vale 1 si dos agrupamientos son idénticos y cerca de 0 si
coinciden como al azar. Se calcula sólo sobre los pozos que DBSCAN no marca como ruido.

In [9]:
pozos = pet.groupby("idpozo").agg(prod_pet=("prod_pet", "sum"), prod_gas=("prod_gas", "sum"),
                                  prod_agua=("prod_agua", "sum"), tef=("tef", "sum"),
                                  profundidad=("profundidad", "first"), tipo_de_recurso=("tipo_de_recurso", "first"),
                                  cuenca=("cuenca", "first"), formprod=("formprod", "first"))
pozos = pozos[(pozos["tef"] > 0) & (pozos["prod_pet"] > 0) & pozos["profundidad"].notna()]
X_c = StandardScaler().fit_transform(pd.DataFrame({
    "log_caudal_pet": np.log1p(pozos["prod_pet"] / pozos["tef"]),
    "log_caudal_gas": np.log1p(pozos["prod_gas"] / pozos["tef"]),
    "corte_agua": pozos["prod_agua"] / (pozos["prod_agua"] + pozos["prod_pet"]),
    "profundidad": pozos["profundidad"]}))
k2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_c)
k3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_c)
alto = pd.Series(k3).groupby(k3).apply(lambda s: np.median((pozos["prod_pet"] / pozos["tef"]).iloc[s.index])).idxmax()
db = DBSCAN(eps=0.6, min_samples=20).fit_predict(X_c)
sin_ruido = db != -1
es_nc = (pozos["tipo_de_recurso"] == "NO CONVENCIONAL").to_numpy()
pd.Series({
    "pozos": len(pozos), "ruido DBSCAN": (~sin_ruido).sum(), "% ruido": (~sin_ruido).mean() * 100,
    "ARI DBSCAN vs K-Means k=2": adjusted_rand_score(db[sin_ruido], k2[sin_ruido]),
    "ARI DBSCAN vs alto caudal (k=3)": adjusted_rand_score(db[sin_ruido], (k3 == alto)[sin_ruido]),
    "ARI alto caudal (k=3) vs no convencional": adjusted_rand_score(es_nc, k3 == alto),
    "% no convencional en alto caudal": es_nc[k3 == alto].mean() * 100,
    "% cuenca Neuquina en alto caudal": (pozos["cuenca"].to_numpy()[k3 == alto] == "NEUQUINA").mean() * 100,
    "% formación VMUT en alto caudal": (pozos["formprod"].to_numpy()[k3 == alto] == "VMUT").mean() * 100,
}).round(3)

pozos                                      21,226.000
ruido DBSCAN                                  723.000
% ruido                                         3.406
ARI DBSCAN vs K-Means k=2                       1.000
ARI DBSCAN vs alto caudal (k=3)                 0.995
ARI alto caudal (k=3) vs no convencional        0.797
% no convencional en alto caudal               92.202
% cuenca Neuquina en alto caudal               93.512
% formación VMUT en alto caudal                92.083
dtype: float64

## 5. Límites que este notebook no resuelve

- **Un solo año.** La validación separa por pozo, no por tiempo: mide cómo generaliza a pozos no vistos de 2025, no a
  2026. No hicimos una prueba temporal.
- **Sin edad ni historia del pozo.** No hay fecha de terminación, así que no se modela la declinación.
- **Asociación, no causalidad.** La importancia de variables y los perfiles muestran asociaciones; no prueban que la
  profundidad, el sistema de extracción o el agua *causen* la producción.